In [1]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from google.colab import drive

In [2]:
# ─── 3. LEITURA E IDENTIFICAÇÃO DAS PLANILHAS ─────────────────────────────────
# ⚙️  Ajuste PASTA_DRIVE para o caminho da pasta no seu Drive onde estão os CSVs.
# Exemplo: 'Meu Drive/dados_acidentes'  ou  'Meu Drive'  se estiverem na raiz.
PASTA_DRIVE = 'Meu Drive'   # ← altere aqui se necessário

ARQUIVOS_ESPERADOS = [
    '2023_famar.csv',
    '2024_famar.csv',
    '2023_fumes.csv',
    '2024_fumes.csv',
]

def identificar_instituicao(nome_arquivo: str) -> str:
    nome = nome_arquivo.lower()
    if 'fumes' in nome:
        return 'FUMES'
    elif 'famar' in nome:
        return 'FAMAR'
    raise ValueError(f'Não foi possível identificar a instituição em: {nome_arquivo}')

planilhas = {}   # {'nome_arquivo': DataFrame}

for nome in ARQUIVOS_ESPERADOS:
    caminho = f'{nome}'
    try:
        df = pd.read_csv(caminho)
    except FileNotFoundError:
        print(f'[AVISO] Arquivo não encontrado: {caminho}')
        continue

    df['instituicao'] = identificar_instituicao(nome)
    df['arquivo_origem'] = nome

    # Converter coluna de data (formato yyyy-dd-MM hh:mm:ss)
    # Tenta converter no formato com hora
    # Tenta converter no formato com hora
    datas_convertidas = pd.to_datetime(
        df['data_do_acidente'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    # Onde falhou, tenta no formato sem hora
    datas_convertidas = datas_convertidas.fillna(
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d',
            errors='coerce'
        )
    )

    # Imprime datas inválidas
    mask_invalidas = (
        datas_convertidas.isna()
        & df['data_do_acidente'].notna()
    )

    if mask_invalidas.any():
        print('\nDatas com erro de conversão:')

        for idx, valor in df.loc[
            mask_invalidas,
            'data_do_acidente'
        ].items():

            print(
                f'  Linha {idx + 2}: '
                f'"{valor}" '
                f'(esperado: %Y-%m-%d %H:%M:%S '
                f'ou %Y-%m-%d)'
            )

    # Salva no dataframe
    df['data_do_acidente'] = datas_convertidas

    planilhas[nome] = df
    print(f'[OK] {nome} — {len(df)} registros | instituição: {df["instituicao"].iloc[0]}')

print(f'\nTotal de arquivos carregados: {len(planilhas)}')

[OK] 2023_famar.csv — 93 registros | instituição: FAMAR
[OK] 2024_famar.csv — 83 registros | instituição: FAMAR
[OK] 2023_fumes.csv — 9 registros | instituição: FUMES
[OK] 2024_fumes.csv — 12 registros | instituição: FUMES

Total de arquivos carregados: 4


In [3]:
# ─── 4. AGREGAÇÃO DOS BIÊNIOS ─────────────────────────────────────────────────
# Biênio FAMAR (2023 + 2024)
df_famar = pd.concat(
    [planilhas[f] for f in ['2023_famar.csv', '2024_famar.csv'] if f in planilhas],
    ignore_index=True
)

# Biênio FUMES (2023 + 2024)
df_fumes = pd.concat(
    [planilhas[f] for f in ['2023_fumes.csv', '2024_fumes.csv'] if f in planilhas],
    ignore_index=True
)

print(f'Biênio FAMAR — total de registros: {len(df_famar)}')
print(f'Biênio FUMES — total de registros: {len(df_fumes)}')

Biênio FAMAR — total de registros: 176
Biênio FUMES — total de registros: 21


In [4]:
import pandas as pd

FORMATO_DATA = '%Y-%m-%d %H:%M:%S'

print('=' * 60)
print('  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA')
print('=' * 60)

total_geral = 0

for nome, df in planilhas.items():
    n = len(df)
    total_geral += n

    inst = (
        df['instituicao'].iloc[0]
        if not df.empty and 'instituicao' in df.columns
        else 'N/D'
    )

    datas_nulas = df['data_do_acidente'].isna().sum()
    datas_invalidas = []

    for idx, val in df['data_do_acidente'].items():

        # Ignora nulos
        if pd.isna(val):
            continue

        linha_planilha = idx + 2  # +1 header +1 índice zero-based
        valor_original = str(val)

        # try:
        #     pd.to_datetime(
        #         valor_original,
        #         format=FORMATO_DATA,
        #         errors='raise'
        #     )

        # except Exception:

        #     motivo = (
        #         'Formato inválido. '
        #         'Esperado: yyyy-mm-dd HH:MM:SS '
        #         f'(ex.: 2023-05-31 00:00:00)'
        #     )

        #     datas_invalidas.append({
        #         'linha': linha_planilha,
        #         'valor': valor_original,
        #         'motivo': motivo
        #     })

    print(f'\nArquivo : {nome}')
    print(f'  Instituição       : {inst}')
    print(f'  Registros totais  : {n}')
    print(f'  Datas nulas       : {datas_nulas}')
    print(f'  Datas inválidas   : {len(datas_invalidas)}')

    if datas_invalidas:
        print('\n  Datas fora do formato:')

        print(
            f'  {"Linha":>6}  '
            f'{"Valor encontrado":<30}  '
            f'Motivo'
        )

        print(
            f'  {"-" * 6}  '
            f'{"-" * 30}  '
            f'{"-" * 60}'
        )

        for erro in datas_invalidas:
            print(
                f'  {erro["linha"]:>6}  '
                f'{erro["valor"]:<30}  '
                f'{erro["motivo"]}'
            )

    else:
        print('  Datas fora de formato : nenhuma')

print(f'\n{" TOTAL GERAL ":=^60}')
print(
    f'  {total_geral} registros importados '
    f'em {len(planilhas)} arquivos'
)

  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA

Arquivo : 2023_famar.csv
  Instituição       : FAMAR
  Registros totais  : 93
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_famar.csv
  Instituição       : FAMAR
  Registros totais  : 83
  Datas nulas       : 1
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2023_fumes.csv
  Instituição       : FUMES
  Registros totais  : 9
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_fumes.csv
  Instituição       : FUMES
  Registros totais  : 12
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

======================= TOTAL GERAL ========================
  197 registros importados em 4 arquivos


In [5]:
# ─── 5. FUNÇÃO: TRIMESTRE A PARTIR DA DATA ────────────────────────────────────
def adicionar_trimestre(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['trimestre'] = (
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        )
        .dt.quarter
        .apply(lambda x: f'T{x}' if pd.notna(x) else None)
    )

    return df


df_famar = adicionar_trimestre(df_famar)
df_fumes = adicionar_trimestre(df_fumes)

print(
    'Trimestres identificados — FAMAR:',
    sorted(df_famar['trimestre'].dropna().unique())
)

print(
    'Trimestres identificados — FUMES:',
    sorted(df_fumes['trimestre'].dropna().unique())
)

Trimestres identificados — FAMAR: ['T1.0', 'T2.0', 'T3.0', 'T4.0']
Trimestres identificados — FUMES: ['T1', 'T2', 'T3', 'T4']


In [6]:
# ─── 7. RELATÓRIO TRIMESTRAL DOS BIÊNIOS ──────────────────────────────────────
def relatorio_trimestral(df: pd.DataFrame, nome_inst: str):
    total = len(df)
    print(f'\n{" " + nome_inst + " — Biênio por Trimestre ":=^60}')
    print(f'  Total do biênio: {total} registros\n')

    por_trim = (
        df.groupby('trimestre', dropna=False)
          .size()
          .reset_index(name='n')
          .sort_values('trimestre')
    )

    print(f'  {"Trimestre":<15} {"N (abs)":>10} {"% do biênio":>14}')
    print(f'  {"-"*15} {"-"*10} {"-"*14}')

    for _, row in por_trim.iterrows():
        trim = str(row['trimestre']) if pd.notna(row['trimestre']) else 'Data inválida'
        n    = int(row['n'])
        pct  = (n / total * 100) if total > 0 else 0
        print(f'  {trim:<15} {n:>10} {pct:>13.1f}%')

    print(f'  {"TOTAL":<15} {total:>10} {100.0:>13.1f}%')

relatorio_trimestral(df_famar, 'FAMAR')
relatorio_trimestral(df_fumes, 'FUMES')


=============== FAMAR — Biênio por Trimestre ===============
  Total do biênio: 176 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1.0                    49          27.8%
  T2.0                    35          19.9%
  T3.0                    52          29.5%
  T4.0                    39          22.2%
  Data inválida            1           0.6%
  TOTAL                  176         100.0%

=============== FUMES — Biênio por Trimestre ===============
  Total do biênio: 21 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1                       5          23.8%
  T2                       6          28.6%
  T3                       7          33.3%
  T4                       3          14.3%
  TOTAL                   21         100.0%


In [17]:
print(df_fumes['parte_do_corpo_atingida'])

0                                                  Dedo
1                                                  Dedo
2                                                Joelho
3     Dorso (inclusive músculos dorsais, coluna e me...
4                                           Cabeça, NIC
5                                                 Punho
6                                                  Dedo
7                                                Joelho
8                                                Joelho
9                    Abdome (inclusive órgãos internos)
10    Quadris (inclusive pélvis, órgãos pélvicos e n...
11    Membros superiores, partes múltiplas (qualquer...
12                 Olho (inclusive nervo ótico e visão)
13                                                 Dedo
14    Partes múltiplas. (aplica-se quando mais de um...
15    Face, partes múltiplas (qualquer combinação da...
16                                                 Dedo
17                                              

In [26]:
# =============================================================================
# ANÁLISE: PARTE DO CORPO ATINGIDA — TRIMESTRE + TESTE DE FISHER
# =============================================================================
# Uso:
#   1. Coloque os arquivos CSV na mesma pasta deste script
#      (ou ajuste PASTA_DADOS abaixo).
#   2. Execute:  python fisher_parte_corpo_trimestre.py
#   3. O resultado é impresso no terminal e salvo em
#      fisher_parte_corpo_resultado.csv
# =============================================================================

import os
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact


# ─── CONFIGURAÇÃO ─────────────────────────────────────────────────────────────

# Pasta onde estão os arquivos CSV. '.' = mesma pasta do script.
PASTA_DADOS = '.'

ARQUIVOS = {
    'FAMAR': ['2023_famar.csv', '2024_famar.csv'],
    'FUMES': ['2023_fumes.csv', '2024_fumes.csv'],
}

# Arquivo de saída
ARQUIVO_SAIDA = 'fisher_parte_corpo_resultado.csv'


# ─── 1. LEITURA E PREPARAÇÃO DOS DADOS ───────────────────────────────────────

def carregar_bienio(instituicao: str, arquivos: list[str]) -> pd.DataFrame:
    """Lê e concatena os CSVs de um biênio; adiciona coluna 'trimestre'."""
    dfs = []
    for nome in arquivos:
        caminho = os.path.join(PASTA_DADOS, nome)
        if not os.path.exists(caminho):
            print(f'[AVISO] Arquivo não encontrado: {caminho}')
            continue

        df = pd.read_csv(caminho)
        df['instituicao']   = instituicao
        df['arquivo_origem'] = nome

        # Conversão de data (aceita com ou sem hora)
        datas = pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        ).fillna(
            pd.to_datetime(df['data_do_acidente'], format='%Y-%m-%d', errors='coerce')
        )
        df['data_do_acidente'] = datas

        # Coluna trimestre: T1 … T4
        df['trimestre'] = (
            datas.dt.quarter
            .apply(lambda x: f'T{int(x)}' if pd.notna(x) else None)
        )

        dfs.append(df)
        print(f'[OK] {nome} — {len(df)} registros')

    if not dfs:
        raise FileNotFoundError(
            f'Nenhum arquivo encontrado para {instituicao}. '
            f'Verifique PASTA_DADOS = "{PASTA_DADOS}".'
        )

    result = pd.concat(dfs, ignore_index=True)
    print(f'Biênio {instituicao} — total: {len(result)} registros\n')
    return result


# ─── 2. MAPEAMENTO EXATO ──────────────────────────────────────────────────────

MAPEAMENTO_EXATO = {
    'artelho': 'PÉS E MM. INFERIORES',

    (
        'sistemas e aparelhos (aplica-se quando o funcionamento de todo um '
        'sistema ou aparelho do corpo humano for afetado, sem lesão específica '
        'de qualquer outra parte, como no caso do envenenamento, ação corrosiva '
        'que afete órgãos internos, lesão dos centros nervosos, etc.; não se '
        'aplica quando a lesão sistêmica for provocada por lesão externa, como '
        'lesão dorsal que afete nervos da medula espinhal)'
    ): 'TRONCO',

    'cabeça, partes múltiplas (qualquer combinação das partes acima)':
        'CABEÇA, FACE E SENTIDOS',

    'pescoço': 'CABEÇA, FACE E SENTIDOS',

    (
        'partes múltiplas. (aplica-se quando mais de uma parte importante do '
        'corpo for afetada, como, por exemplo, um braço e uma perna)'
    ): 'PARTES MÚLTIPLAS',
}


# ─── 3. TERMOS PARA BUSCA PARCIAL ────────────────────────────────────────────

TERMOS = {
    'MÃOS E MM. SUPERIORES': [
        'mão', 'mãos', 'mao', 'maos', 'dedo', 'dedos', 'polegar',
        'indicador', 'punho', 'pulso', 'antebraço', 'antebraco',
        'cotovelo', 'braco', 'braço', 'ombro', 'membro superior',
        'membros superiores', 'mm superior', 'mm. superior',
        'extremidade superior', 'axila', 'palma', 'falange',
    ],
    'PÉS E MM. INFERIORES': [
        'artelho', 'pe', 'pé', 'pes', 'pés', 'tornozelo',
        'calcanhar', 'calcaneo', 'calcâneo', 'halux',
        'haluce', 'dedos do pe', 'dedos do pé',
        'perna', 'joelho', 'coxa', 'quadril',
        'membro inferior', 'membros inferiores',
        'mm inferior', 'mm. inferior',
        'extremidade inferior', 'plantar',
        'tibio', 'tibia', 'tíbia',
        'fibula', 'fíbula',
    ],
    'CABEÇA, FACE E SENTIDOS': [
        'cabeca', 'cabeça', 'cranio', 'crânio',
        'face', 'rosto', 'olho', 'olhos',
        'ouvido', 'ouvidos', 'orelha',
        'nariz', 'boca', 'dente', 'dentes',
        'mento', 'mandibula', 'mandíbula',
        'maxilar', 'fronte', 'testa',
        'nuca', 'pescoco', 'pescoço',
        'cervical', 'visao', 'visão',
        'audicao', 'audição',
        'sentido', 'sentidos',
        'orbita', 'órbita',
    ],
    'TRONCO': [
        'tronco', 'torax', 'tórax',
        'torace', 'costela', 'costelas',
        'peito', 'abdome', 'abdomen',
        'abdômen', 'lombar', 'lombo',
        'coluna', 'dorsal', 'dorso',
        'costas', 'escapula', 'escápula',
        'clavícula', 'clavicula',
        'esterno', 'pelvico', 'pélvico',
        'pelve', 'bacia', 'virilha',
        'inguinal', 'sacro', 'coccix',
        'cóccix', 'cervico',
        'cervico-dorsal',
        'sistema', 'aparelho',
        'orgao', 'órgão',
        'viscera', 'víscera',
    ],
    # sempre último (mais genérico)
    'PARTES MÚLTIPLAS': [
        'partes múltiplas', 'partes multiplas',
        'multipla', 'múltipla', 'multiplas', 'múltiplas',
        'varias partes', 'várias partes',
        'corpo inteiro', 'todo o corpo',
        'generalizado', 'politraumatismo',
    ],
}

ORDEM_TERMOS = [
    'MÃOS E MM. SUPERIORES',
    'PÉS E MM. INFERIORES',
    'CABEÇA, FACE E SENTIDOS',
    'TRONCO',
    'PARTES MÚLTIPLAS',
]

CATEGORIAS = ORDEM_TERMOS + ['IGNORADO']


# ─── 4. CLASSIFICADOR ─────────────────────────────────────────────────────────

def classificar_parte(valor) -> str:
    if pd.isna(valor):
        return 'IGNORADO'
    v_lower = str(valor).strip().lower()
    if not v_lower:
        return 'IGNORADO'

    # 1. mapeamento exato
    for chave, categoria in MAPEAMENTO_EXATO.items():
        if v_lower == chave.lower():
            return categoria

    # 2. busca parcial (ordem importa)
    for categoria in ORDEM_TERMOS:
        if any(termo in v_lower for termo in TERMOS[categoria]):
            return categoria

    # 3. fallback
    return 'IGNORADO'


# ─── 5. TESTE DE FISHER POR TRIMESTRE ────────────────────────────────────────

def fisher_parte_corpo_trimestre(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """
    Para cada combinação (trimestre × categoria), monta a tabela 2×2 e
    aplica o teste exato de Fisher.

    Tabela 2×2:
        ┌─────────────────────────┬────────────────────────────┐
        │  categoria no trimestre │  outras categorias trim.   │  ← trimestre focal
        ├─────────────────────────┼────────────────────────────┤
        │  categoria fora trim.   │  outras categorias fora    │  ← demais trimestres
        └─────────────────────────┴────────────────────────────┘
    """
    df = df.copy()
    df['categoria_pc'] = df['parte_do_corpo_atingida'].apply(classificar_parte)

    resultados = []

    for trimestre in ['T1', 'T2', 'T3', 'T4']:
        mask_trim = df['trimestre'] == trimestre
        df_trim   = df[mask_trim]
        total_trim = len(df_trim)

        for categoria in CATEGORIAS:
            # Contagens para a tabela 2×2
            a = (df_trim['categoria_pc'] == categoria).sum()          # trimestre & categoria
            b = (df_trim['categoria_pc'] != categoria).sum()          # trimestre & ~categoria
            c = (~mask_trim & (df['categoria_pc'] == categoria)).sum() # ~trimestre & categoria
            d = (~mask_trim & (df['categoria_pc'] != categoria)).sum() # ~trimestre & ~categoria

            pct = (a / total_trim * 100) if total_trim > 0 else 0.0
            total_categoria = (df['categoria_pc'] == categoria).sum()

            try:
                or_val, p_val = fisher_exact([[a, b], [c, d]])
            except Exception:
                or_val, p_val = np.nan, np.nan

            sig = (
                '✔✔✔' if p_val < 0.001 else
                '✔✔'  if p_val < 0.01  else
                '✔'   if p_val < 0.05  else
                '✗'
            )

            resultados.append({
                'Instituição':           nome,
                'Trimestre':             trimestre,
                'Categoria':             categoria,
                'N':                     int(a),
                '%':                     round(pct, 1),
                'Total categoria biênio': int(total_categoria),
                'Total trimestre':       int(total_trim),
                'OR':                    round(or_val, 4) if not np.isnan(or_val) else np.nan,
                'p-valor Fisher':        round(p_val, 6)  if not np.isnan(p_val)  else np.nan,
                'Sig':                   sig,
            })

    resultado = pd.DataFrame(resultados)

    print(f"\n{'=' * 110}")
    print(f"{nome} — PARTE DO CORPO ATINGIDA POR TRIMESTRE")
    print(f"{'=' * 110}")
    print(resultado.to_string(index=False))

    return resultado


# ─── 6. EXECUÇÃO PRINCIPAL ───────────────────────────────────────────────────

if __name__ == '__main__':
    print('\n── Carregando dados ──────────────────────────────────────────')
    df_famar = carregar_bienio('FAMAR', ARQUIVOS['FAMAR'])
    df_fumes = carregar_bienio('FUMES', ARQUIVOS['FUMES'])

    print('\n── Rodando Fisher ────────────────────────────────────────────')
    resultado_famar = fisher_parte_corpo_trimestre(df_famar, 'FAMAR')
    resultado_fumes = fisher_parte_corpo_trimestre(df_fumes, 'FUMES')

    # Combina e salva
    resultado_final = pd.concat([resultado_famar, resultado_fumes], ignore_index=True)
    resultado_final.to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8-sig')
    print(f'\n[✓] Resultado salvo em: {ARQUIVO_SAIDA}')


── Carregando dados ──────────────────────────────────────────
[OK] 2023_famar.csv — 93 registros
[OK] 2024_famar.csv — 83 registros
Biênio FAMAR — total: 176 registros

[OK] 2023_fumes.csv — 9 registros
[OK] 2024_fumes.csv — 12 registros
Biênio FUMES — total: 21 registros


── Rodando Fisher ────────────────────────────────────────────

FAMAR — PARTE DO CORPO ATINGIDA POR TRIMESTRE
Instituição Trimestre               Categoria  N    %  Total categoria biênio  Total trimestre     OR  p-valor Fisher Sig
      FAMAR        T1   MÃOS E MM. SUPERIORES 25 51.0                      92               49 0.9328        0.867487   ✗
      FAMAR        T1    PÉS E MM. INFERIORES  8 16.3                      21               49 1.7111        0.301520   ✗
      FAMAR        T1 CABEÇA, FACE E SENTIDOS 11 22.4                      29               49 1.7529        0.255948   ✗
      FAMAR        T1                  TRONCO  2  4.1                       6               49 1.3085        0.671082   ✗
   